# DGD Test Inference (Algorithm 2)

Loads the frozen `best/` decoder + GMM and optimizes a fresh representation layer for the genuinely held-out test split -- the first real use of `test_loader` anywhere in this codebase.

**Note:** the optimization below applies the same latent-space noise regularization (`training.latent_noise_scale`/`training.latent_noise_start`/`training.latent_noise_end`) that training uses, annealed over the `M` optimization steps instead of epochs. This is a deliberate deviation from a textbook-clean MAP estimate of `z`: the frozen decoder and GMM were themselves trained under noise, so evaluating against a clean (noise-free) `z` would silently shift the operating point relative to what the model was optimized for.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn.functional as F

from omegaconf import OmegaConf, open_dict
from hydra import initialize, compose

current_dir = Path.cwd()
if 'notebooks' in current_dir.parts:
    project_root = current_dir.parent
else:
    project_root = current_dir

sys.path.append(str(project_root))
sys.path.append(str(project_root / 'src'))

from src.data import create_dataloaders, collect_all_labels, collect_class_samples
from src.models import RepresentationLayer, ConvDecoder
from src.utils import setup_device, set_random_seed, setup_cuml_acceleration
from src.utils.checkpoint import load_checkpoint
from src.visualization import generate_inference_figures

device = setup_device(verbose=True)
set_random_seed(seed=42, device=device)
setup_cuml_acceleration(verbose=True)


In [ ]:
config.data.root_dir = str(project_root / "data")
config.paths.experiments_dir = str(project_root / "experiments")

experiments_dir = Path(config.paths.experiments_dir)
candidates = sorted(
    p for p in experiments_dir.glob(f"*_{config.experiment_name}")
    if (p / "models" / "best").is_dir()
)
assert candidates, (
    f"No completed training runs found under {experiments_dir} matching "
    f"*_{config.experiment_name} (looked for a models/best/ subfolder). "
    "Run dgd_training_demo.ipynb first."
)
run_dir = candidates[-1]  # newest, since the timestamp prefix sorts lexicographically
print(f"Using training run: {run_dir}")

# hydra's compose() returns a struct-mode config: models_dir isn't in
# config.yaml's paths: block (only experiments_dir is, since it's resolved
# per-run rather than being a static default), so assigning it requires
# open_dict() to temporarily allow a new key -- a plain assignment would
# raise ConfigAttributeError: Key 'models_dir' is not in struct.
with open_dict(config):
    config.paths.models_dir = str(run_dir / "models")

# Guard: make sure this notebook's config matches the config actually used
# during training, so `test_loader` below is exactly the held-out split
# carved out at training time -- not a silently different split produced by
# a config.yaml that has since changed (random_seed, subset fraction, or
# split ratios).
trained_cfg = OmegaConf.load(run_dir / "config.yaml")
assert config.random_seed == trained_cfg.random_seed, (
    f"random_seed differs from the training run ({config.random_seed} vs {trained_cfg.random_seed}); "
    "the test split would not match the one carved out during training."
)
for key in ['total_subset_fraction', 'val_split', 'test_split']:
    assert config.data[key] == trained_cfg.data[key], (
        f"data.{key} differs from the training run ({config.data[key]} vs {trained_cfg.data[key]}); "
        "the test split would not match the one carved out during training."
    )

# Re-derive the same 3-way split independently (same random_seed as training),
# so `test_loader` here is exactly the held-out split carved out during training
# and never optimized against.
train_loader, val_loader, test_loader, class_names = create_dataloaders(config)
print(f"Test loader: {len(test_loader)} batches, {len(test_loader.dataset)} samples")


In [ ]:
def decoder_factory():
    return ConvDecoder(
        latent_dim=config.model.representation.n_features,
        hidden_dims=config.model.decoder.hidden_dims,
        output_channels=config.model.decoder.output_channels,
        output_size=config.model.decoder.output_size,
        activation=config.model.decoder.activation,
        final_activation=config.model.decoder.final_activation,
        dropout_rate=config.model.decoder.dropout_rate,
        init_size=config.model.decoder.init_size,
    )

best_dir = Path(config.paths.models_dir) / "best"
checkpoint = load_checkpoint(best_dir, decoder_factory, device=device)

decoder = checkpoint['decoder']
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

gmm = checkpoint['gmm']
if gmm is None:
    raise RuntimeError(
        f"No fitted GMM found in {best_dir} -- was training.first_epoch_gmm ever reached "
        "during training? Re-run dgd_training_demo.ipynb first."
    )

meta = checkpoint['metadata']
print(f"Loaded frozen decoder + GMM from {best_dir} "
      f"(best_epoch={meta.get('best_epoch')}, best_val_loss={meta.get('best_val_loss'):.4f})")


In [ ]:
# Algorithm 2, line 1: initialize a fresh representation layer for the new
# (held-out) data, using the same generic init distribution training used for
# Z_0 -- not a GMM-sample init, matching how the codebase already initializes
# val_rep alongside rep in DGDTrainer._create_model_components.
model_config = config.model
distribution = model_config.representation.distribution

if distribution == 'pca':
    print("Representation distribution is 'pca'; using 'normal' for the test layer "
          "instead (matches how val_rep is initialized under PCA during training).")
    test_distribution = 'normal'
    dist_params = {}
else:
    test_distribution = distribution
    dist_params = {}
    if hasattr(model_config.representation, 'radius'):
        dist_params['radius'] = model_config.representation.radius
    for param in ['mean', 'cov', 'low', 'high', 'loc', 'scale', 'scale_matrix',
                 'rate', 'df', 'mu', 'alpha', 'beta', 'delta']:
        if hasattr(model_config.representation, param):
            dist_params[param] = getattr(model_config.representation, param)

test_rep = RepresentationLayer(
    dim=model_config.representation.n_features,
    n_samples=len(test_loader.dataset),
    dist=test_distribution,
    dist_params=dist_params,
    device=device,
)
print(f"Initialized test representation layer: {test_rep.n_rep} samples x {test_rep.dim} dims")


In [ ]:
# Algorithm 2, lines 2-5: optimize test_rep alone against the frozen decoder+GMM.
# M0 (prior_warmup_steps) < M (epochs): reconstruction-only warm-up before the
# GMM prior term is added, so a fresh z isn't dominated by the prior gradient
# before it has any reconstruction signal to work with.
import math

lr_config = config.training.lr_scheduler
rep_config = config.training.optimizer.representation

test_optimizer = torch.optim.AdamW(
    test_rep.parameters(),
    lr=rep_config.lr,
    betas=tuple(rep_config.betas),
    eps=rep_config.eps,
    weight_decay=rep_config.weight_decay,
    amsgrad=rep_config.get('amsgrad', False),
)

M = config.training.inference.epochs
M0 = config.training.inference.prior_warmup_steps
assert M0 < M, "training.inference.prior_warmup_steps must be < training.inference.epochs"

if lr_config.get('enabled', False):
    max_lr = lr_config.get('max_lr_representation', None) or rep_config.lr
    test_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        test_optimizer,
        max_lr=max_lr,
        total_steps=M,
        pct_start=lr_config.get('pct_start', 0.3),
        anneal_strategy=lr_config.get('anneal_strategy', 'cos'),
        div_factor=lr_config.get('div_factor', 25.0),
        final_div_factor=lr_config.get('final_div_factor', 10000.0),
        cycle_momentum=lr_config.get('cycle_momentum', True),
        base_momentum=lr_config.get('base_momentum', 0.85),
        max_momentum=lr_config.get('max_momentum', 0.95),
        three_phase=lr_config.get('three_phase', False),
    )
else:
    test_scheduler = None

lambda_gmm = config.training.lambda_gmm
n_test = len(test_loader.dataset)

# Inference has its own noise schedule, independent of training's
# latent_noise_scale/_start/_end -- same cosine-anneal formula, mapped onto
# step index m instead of epoch.
latent_noise_scale = config.training.inference.get('latent_noise_scale', 0.0)
noise_start = config.training.inference.get('latent_noise_start', 1.0)
noise_end = config.training.inference.get('latent_noise_end', 0.01)

print(f"Optimizing {n_test} test representations for {M} steps (prior warm-up: {M0} steps)...")

step_history = {'loss': [], 'recon': [], 'gmm': [], 'noise': []}

for m in range(1, M + 1):
    test_optimizer.zero_grad()

    if latent_noise_scale > 0:
        progress = (m - 1) / max(M - 1, 1)
        noise_scale_m = noise_end + (noise_start - noise_end) * 0.5 * (1 + math.cos(math.pi * progress))
    else:
        noise_scale_m = 0.0

    total_loss = 0.0
    total_recon = 0.0
    total_gmm = 0.0

    for index, x, _ in test_loader:
        x, index = x.to(device), index.to(device)

        z = test_rep(index)

        if noise_scale_m > 0:
            z = z + torch.randn_like(z) * noise_scale_m

        y = decoder(z)
        recon_loss = F.mse_loss(y, x, reduction='sum')

        if m >= M0:
            gmm_error = -lambda_gmm * torch.sum(gmm.score_samples(z))
            loss = recon_loss + gmm_error
        else:
            gmm_error = torch.tensor(0.0, device=device)
            loss = recon_loss

        loss.backward()

        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_gmm += gmm_error.item()

    test_optimizer.step()
    if test_scheduler is not None:
        test_scheduler.step()

    step_history['loss'].append(total_loss / n_test)
    step_history['recon'].append(total_recon / n_test)
    step_history['gmm'].append(total_gmm / n_test)
    step_history['noise'].append(noise_scale_m)

    if m % max(1, M // 10) == 0 or m == M:
        print(f"Step {m}/{M}: loss={total_loss/n_test:.4f}, recon={total_recon/n_test:.4f}, gmm={total_gmm/n_test:.4f}, noise={noise_scale_m:.4f}")

print("Test representation optimization complete.")


In [ ]:
from datetime import datetime

inference_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
inference_dir = run_dir / "inference" / inference_timestamp
inference_dir.mkdir(parents=True, exist_ok=True)

OmegaConf.save(config, str(inference_dir / "config.yaml"))

test_rep.save(str(inference_dir / "test_representation.pt"))
print(f"Saved optimized test representations to {inference_dir / 'test_representation.pt'}")


In [ ]:
figures_dir = inference_dir / "figures"

test_labels = collect_all_labels(test_loader)

sample_data = collect_class_samples(test_loader, n_per_class=5, n_classes=len(class_names))

step_history_totals = step_history

test_ami, test_ari = generate_inference_figures(
    figures_dir=figures_dir,
    decoder=decoder,
    gmm=gmm,
    test_rep=test_rep,
    test_labels=test_labels,
    class_names=class_names,
    sample_data=sample_data,
    step_history=step_history_totals,
    device=device,
)

print(f"Test AMI: {test_ami:.4f}, Test ARI: {test_ari:.4f}")
print(f"Figures written to {figures_dir}")
